#### 1.Call Liberary

In [1]:
import os
import requests
import pandas as pd
import re
import json
from dotenv import load_dotenv

#### 2.Load Variable env from dotenv to get "API_KEY"

In [2]:
# Charger les variables d'environnement
load_dotenv()
API_KEY = os.getenv("YOUTUBE_API_KEY")

#### 3.channel YOUTUBE Name we need to handle 

In [3]:
CHANNEL_HANDLE = "MEDINCODEX" 

#### 4.Function to get Channel Info

In [4]:
def get_channel_info_by_handle(api_key, handle):
    url = "https://www.googleapis.com/youtube/v3/channels"
    params = {"part": "snippet,contentDetails,statistics", 
              "forHandle": handle, 
              "key": api_key}
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        # print(f"Data received for channel handle '{handle}': {data}")
        if "items" in data and len(data["items"]) > 0:
            return data["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]
    return None
#show the channel id
channel_id = get_channel_info_by_handle(API_KEY, CHANNEL_HANDLE)
print(f"Channel ID for {CHANNEL_HANDLE}: {channel_id}")

Channel ID for MEDINCODEX: UUS65-lIOuLB0MsT4Gnv94dA


#### 5.Function to get all video from playlist

In [5]:
def get_video_ids(api_key, playlist_id):
    video_ids = []
    url = "https://www.googleapis.com/youtube/v3/playlistItems"
    next_page_token = None
    print(" Extraction des IDs de vidéos en cours...")
    while True:
        params = {"part": "contentDetails", "playlistId": playlist_id, "maxResults": 50, "pageToken": next_page_token, "key": api_key}
        response = requests.get(url, params=params)
        if response.status_code == 200:
            data = response.json()
            for item in data.get("items", []):
                video_ids.append(item["contentDetails"]["videoId"])
            next_page_token = data.get("nextPageToken")
            if not next_page_token:
                break
        else:
            break
    return video_ids

#### 6.Function to get Video details

In [6]:
def get_video_details(api_key, video_ids):
    all_video_stats = []
    url = "https://www.googleapis.com/youtube/v3/videos"
    print(" Extraction des détails des vidéos en cours (Batching)...")
    for i in range(0, len(video_ids), 50):
        batch_ids = video_ids[i:i+50]
        ids_string = ",".join(batch_ids)
        params = {"part": "snippet,contentDetails,statistics", "id": ids_string, "key": api_key}
        response = requests.get(url, params=params)
        if response.status_code == 200:
            data = response.json()
            for item in data.get("items", []):
                stats = {
                    "video_id": item["id"],
                    "title": item["snippet"]["title"],
                    "published_at": item["snippet"]["publishedAt"],
                    "duration": item["contentDetails"]["duration"],
                    "views": item["statistics"].get("viewCount", 0),
                    "likes": item["statistics"].get("likeCount", 0),
                    "comments": item["statistics"].get("commentCount", 0)
                }
                all_video_stats.append(stats)
    return all_video_stats

#### 7.Function to transform Time ISO to Time HH:MM:SS

In [7]:
def parse_duration(duration):
    hours_match = re.search(r'(\d+)H', duration)
    minutes_match = re.search(r'(\d+)M', duration)
    seconds_match = re.search(r'(\d+)S', duration)
    
    hours = int(hours_match.group(1)) if hours_match else 0
    minutes = int(minutes_match.group(1)) if minutes_match else 0
    seconds = int(seconds_match.group(1)) if seconds_match else 0
    
    return hours * 3600 + minutes * 60 + seconds
def transform_data(video_data):
    print(" Transformation des données en cours...")
    df = pd.DataFrame(video_data)
    df['views'] = pd.to_numeric(df['views']).fillna(0).astype(int)
    df['likes'] = pd.to_numeric(df['likes']).fillna(0).astype(int)
    df['comments'] = pd.to_numeric(df['comments']).fillna(0).astype(int)
    df['published_at'] = pd.to_datetime(df['published_at'])
    df['publish_date'] = df['published_at'].dt.date
    df['publish_time'] = df['published_at'].dt.time
    df['duration_seconds'] = df['duration'].apply(parse_duration)
    df = df.drop(columns=['published_at', 'duration'])
    return df

#### 8.Function to transform donnée Brute to DataFrame & CSV

In [8]:
def save_data_to_json(df, filename="data/youtube_data.json"):
    """Sauvegarde le DataFrame dans un fichier JSON."""
    print(f"\n Sauvegarde des données dans {filename}...")
    os.makedirs("data", exist_ok=True)
    df.to_json(filename, orient="records", indent=4, force_ascii=False)
    print(" Succès ! Les données JSON sont prêtes et sauvegardées.")

def save_data_to_csv(df, filename="data/youtube_data.csv"):
    """Sauvegarde le DataFrame dans un fichier CSV."""
    print(f"Sauvegarde des données dans {filename}...")
    
    # Créer le dossier 'data' s'il n'existe pas (par précaution)
    os.makedirs("data", exist_ok=True)
    
    # Sauvegarder en CSV (utf-8-sig pour éviter les problèmes d'accents sur Windows/Excel)
    df.to_csv(filename, index=False, encoding='utf-8-sig')
    print(" Succès ! Les données sont prêtes et sauvegardées.")

if __name__ == "__main__":
    if API_KEY:
        uploads_id = get_channel_info_by_handle(API_KEY, CHANNEL_HANDLE)
        if uploads_id:
            videos_list = get_video_ids(API_KEY, uploads_id)
            if videos_list:
                video_data = get_video_details(API_KEY, videos_list)
                if video_data:
                    # 1. Transformer les données
                    df_final = transform_data(video_data)
                    
                    # 2. Sauvegarder les données en CSV
                    save_data_to_csv(df_final, f"data/youtube_data_{CHANNEL_HANDLE.lower()}.csv")

                    # 3. Sauvegarder les données en JSON
                    save_data_to_json(df_final, f"data/youtube_data_{CHANNEL_HANDLE.lower()}.json")       
    else:
        print(" Clé API introuvable.")     

 Extraction des IDs de vidéos en cours...
 Extraction des détails des vidéos en cours (Batching)...
 Transformation des données en cours...
Sauvegarde des données dans data/youtube_data_medincodex.csv...
 Succès ! Les données sont prêtes et sauvegardées.

 Sauvegarde des données dans data/youtube_data_medincodex.json...
 Succès ! Les données JSON sont prêtes et sauvegardées.


In [9]:
df=pd.read_json("data/youtube_data_medincodex.json")
df["publish_date"] = pd.to_datetime(df["publish_date"], unit="ms")
df

,video_id,title,views,likes,comments,publish_date,publish_time,duration_seconds
0,uAxnTXucNTg,سورة نوح دعوة للتوحيد وثبات على الحق وكيف يُبن...,11,0,0,2025-05-02,2026-04-29 05:59:54,262
1,eHSR1E7gV-4,سورة المعارج عروج الأرواح وارتقاء الإنسان في ط...,6,1,0,2025-04-27,2026-04-29 20:59:09,204
2,p7KzsaO2M5o,سورة الحاقة:تلاوة مؤثرة وتصوير مهيب ليوم القيا...,12,1,0,2025-03-04,2026-04-29 04:59:24,462
3,6EljsjZ09xE,سورة القلم بين الإيمان والضلال، موازين العدل ...,5,0,0,2025-03-02,2026-04-29 16:16:17,354
4,uAPgCqm7ifQ,سورة الملك نور في الظلمات ورفيق القبر التي تحم...,15,1,0,2025-01-26,2026-04-29 14:06:55,395
...,...,...,...,...,...,...,...,...
166,mlsuv0ZmofY,💎💎شحن الوك 👑مجانا للمتابعين وتوزيع💎💎 5 فاير با...,167,28,8,2020-05-15,2026-04-29 16:22:00,9903
167,Z31ap_UiMlQ,BOOYAH! تطبيق من ذهب فرصتك لربح العديد من الجو...,888,90,15,2020-05-15,2026-04-29 11:51:18,170
168,SpVCHFidaCQ,فري فاير تروبل في البيك وبرازيليا بحثا عن المج...,173,54,13,2020-05-13,2026-04-29 08:35:14,252
169,w86QiA7rOVg,Trouble in troubles when a hacker knock me dow...,191,49,8,2020-05-05,2026-04-29 03:46:07,168
